In [ ]:
import os
from dotenv import load_dotenv

from youtube_transcript_api import YouTubeTranscriptApi

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# ============================================================
# 1. LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("\n" + "=" * 65)
    print("ERROR: OPENAI_API_KEY NOT FOUND")
    print("=" * 65)
    print(
        "\nPlease create a .env file in the project folder "
        "and add:\n"
    )
    print("OPENAI_API_KEY=your_actual_openai_api_key")
    print()
    raise SystemExit(1)


# ============================================================
# 2. APPLICATION CONFIGURATION
# ============================================================

VIDEO_ID = "hXb1k59w3M8"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
RETRIEVAL_K = 4

EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

VECTOR_DB_PATH = f"./youtube_rag_db_{VIDEO_ID}"


# ============================================================
# 3. DISPLAY HEADER
# ============================================================

print("\n")
print("=" * 65)
print("                 YOUTUBE AI ASSISTANT")
print("=" * 65)
print()
print(f"Video ID       : {VIDEO_ID}")
print(f"Embedding      : {EMBEDDING_MODEL}")
print(f"LLM            : {LLM_MODEL}")
print(f"Chunk Size     : {CHUNK_SIZE}")
print(f"Chunk Overlap  : {CHUNK_OVERLAP}")
print("=" * 65)


# ============================================================
# 4. FETCH YOUTUBE TRANSCRIPT
# ============================================================

def fetch_transcript(video_id):

    print("\n[1/6] Fetching YouTube transcript...")

    try:

        youtube_api = YouTubeTranscriptApi()

        transcript = youtube_api.fetch(video_id)

        full_text = " ".join(
            item.text
            for item in transcript
        )

        if not full_text.strip():

            raise ValueError(
                "The YouTube transcript is empty."
            )

        print("      ✓ Transcript fetched successfully.")

        print(
            f"      ✓ Transcript length: "
            f"{len(full_text):,} characters"
        )

        return full_text

    except Exception as error:

        print("\n      ✗ Unable to fetch transcript.")
        print(f"\n      Error: {error}")

        print(
            "\nPossible reasons:"
        )

        print(
            "      • Invalid YouTube Video ID"
        )

        print(
            "      • Transcript/captions unavailable"
        )

        print(
            "      • YouTube blocked transcript access"
        )

        print(
            "      • Video does not have captions"
        )

        raise SystemExit(1)


# ============================================================
# 5. CREATE DOCUMENT
# ============================================================

def create_document(full_text, video_id):

    return Document(
        page_content=full_text,
        metadata={
            "video_id": video_id,
            "source": f"https://www.youtube.com/watch?v={video_id}"
        }
    )


# ============================================================
# 6. SPLIT TRANSCRIPT
# ============================================================

def split_transcript(document):

    print("\n[2/6] Splitting transcript into chunks...")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    documents = text_splitter.split_documents(
        [document]
    )

    print(
        f"      ✓ Created {len(documents)} chunks."
    )

    return documents


# ============================================================
# 7. CREATE EMBEDDINGS
# ============================================================

def create_embeddings():

    print("\n[3/6] Initializing OpenAI embeddings...")

    embeddings = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        api_key=OPENAI_API_KEY
    )

    print(
        "      ✓ Embedding model ready."
    )

    return embeddings


# ============================================================
# 8. CREATE CHROMA VECTOR DATABASE
# ============================================================

def create_vectorstore(documents, embeddings):

    print("\n[4/6] Creating Chroma vector database...")

    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH
    )

    print(
        f"      ✓ Vector database ready."
    )

    print(
        f"      ✓ Database path: {VECTOR_DB_PATH}"
    )

    return vectorstore


# ============================================================
# 9. CREATE RETRIEVER
# ============================================================

def create_retriever(vectorstore):

    retriever = vectorstore.as_retriever(
        search_kwargs={
            "k": RETRIEVAL_K
        }
    )

    print(
        f"      ✓ Retriever configured "
        f"with top {RETRIEVAL_K} chunks."
    )

    return retriever


# ============================================================
# 10. CREATE RAG PROMPT
# ============================================================

def create_prompt():

    return ChatPromptTemplate.from_template(
        """
You are an AI assistant that answers questions
about a YouTube video.

Your answer must be based ONLY on the retrieved
YouTube transcript context.

Follow these rules:

1. Do not use outside knowledge.
2. Do not invent information.
3. Answer directly and clearly.
4. Use simple, natural language.
5. If the answer is not available in the transcript,
   say:

"I don't know based on the video transcript."

Retrieved transcript context:
{context}

User question:
{question}

Answer:
"""
    )


# ============================================================
# 11. CREATE LLM
# ============================================================

def create_llm():

    print("\n[5/6] Initializing AI model...")

    llm = ChatOpenAI(
        model=LLM_MODEL,
        temperature=0,
        api_key=OPENAI_API_KEY
    )

    print(
        f"      ✓ {LLM_MODEL} ready."
    )

    return llm


# ============================================================
# 12. FORMAT RETRIEVED DOCUMENTS
# ============================================================

def format_docs(documents):

    return "\n\n".join(
        document.page_content
        for document in documents
    )


# ============================================================
# 13. BUILD RAG CHAIN
# ============================================================

def create_rag_chain(retriever, prompt, llm):

    rag_chain = (
        {
            "context": (
                retriever
                | format_docs
            ),
            "question": (
                RunnablePassthrough()
            )
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print("\n[6/6] Building RAG pipeline...")

    print(
        "      ✓ Retriever connected."
    )

    print(
        "      ✓ Prompt connected."
    )

    print(
        "      ✓ LLM connected."
    )

    print(
        "      ✓ Output parser connected."
    )

    return rag_chain


# ============================================================
# 14. ASK QUESTION
# ============================================================

def ask_question(rag_chain, question):

    try:

        response = rag_chain.invoke(
            question
        )

        return response

    except Exception as error:

        print(
            "\n✗ Unable to generate answer."
        )

        print(
            f"\nError: {error}"
        )

        return None


# ============================================================
# 15. MAIN APPLICATION
# ============================================================

def main():

    # --------------------------------------------------------
    # FETCH TRANSCRIPT
    # --------------------------------------------------------

    full_text = fetch_transcript(
        VIDEO_ID
    )

    # --------------------------------------------------------
    # CREATE DOCUMENT
    # --------------------------------------------------------

    document = create_document(
        full_text,
        VIDEO_ID
    )

    # --------------------------------------------------------
    # SPLIT TRANSCRIPT
    # --------------------------------------------------------

    documents = split_transcript(
        document
    )

    # --------------------------------------------------------
    # EMBEDDINGS
    # --------------------------------------------------------

    embeddings = create_embeddings()

    # --------------------------------------------------------
    # VECTOR STORE
    # --------------------------------------------------------

    vectorstore = create_vectorstore(
        documents,
        embeddings
    )

    # --------------------------------------------------------
    # RETRIEVER
    # --------------------------------------------------------

    print("\n      Configuring retriever...")

    retriever = create_retriever(
        vectorstore
    )

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    prompt = create_prompt()

    # --------------------------------------------------------
    # LLM
    # --------------------------------------------------------

    llm = create_llm()

    # --------------------------------------------------------
    # RAG CHAIN
    # --------------------------------------------------------

    rag_chain = create_rag_chain(
        retriever,
        prompt,
        llm
    )

    # --------------------------------------------------------
    # READY MESSAGE
    # --------------------------------------------------------

    print("\n")
    print("=" * 65)
    print("                 ✓ RAG SYSTEM READY")
    print("=" * 65)

    print(
        "\nYou can now ask questions about the video."
    )

    print(
        "Type 'exit' to close the application."
    )

    print(
        "Type 'clear' to clear the terminal screen."
    )

    print("=" * 65)

    # --------------------------------------------------------
    # QUESTION LOOP
    # --------------------------------------------------------

    while True:

        try:

            question = input(
                "\nYou: "
            ).strip()

        except KeyboardInterrupt:

            print(
                "\n\nApplication closed."
            )

            break

        # ----------------------------------------------------
        # EMPTY INPUT
        # ----------------------------------------------------

        if not question:

            print(
                "Please enter a question."
            )

            continue

        # ----------------------------------------------------
        # EXIT
        # ----------------------------------------------------

        if question.lower() in {
            "exit",
            "quit",
            "q"
        }:

            print(
                "\nThank you for using "
                "YouTube AI Assistant."
            )

            break

        # ----------------------------------------------------
        # CLEAR
        # ----------------------------------------------------

        if question.lower() == "clear":

            os.system(
                "cls"
                if os.name == "nt"
                else "clear"
            )

            print(
                "YouTube AI Assistant"
            )

            print(
                f"Video: {VIDEO_ID}"
            )

            continue

        # ----------------------------------------------------
        # GENERATE ANSWER
        # ----------------------------------------------------

        print(
            "\nSearching transcript..."
        )

        response = ask_question(
            rag_chain,
            question
        )

        if response:

            print("\nAI:")
            print("-" * 65)
            print(response)
            print("-" * 65)


# ============================================================
# 16. APPLICATION ENTRY POINT
# ============================================================

if __name__ == "__main__":

    main()

Fetching transcript...
Transcript loaded successfully.
Split into 32 chunks.
Vector store created.
RAG system ready.

Answer:

The agenda of the meeting includes having conversations about technology, specifically focusing on AI, robotics, energy, and space. The goal is to discuss the meaningful components of these technologies, their engineering challenges, and how they can maximize the future of civilization and expand consciousness beyond Earth. Additionally, there is an emphasis on understanding and resolving various issues through dialogue.

Answer:

I don't know.

Answer:

This discussion is happening in Davos.

Answer:

1. AI Progression: The speaker predicts that AI may surpass human intelligence by the end of this year or next year, and by 2030 or 2031, AI could be smarter than all of humanity collectively.

2. Inspiration and Curiosity: The speaker reflects on their childhood interest in science fiction as a foundation for their curiosity and innovation.

3. Investment Potent